
# Train-Test Splits and Cross-Validation

The ultimate goal of a machine learning model is not to memorize data, but to **generalize**. Imagine a student preparing for a math exam:

-   If they memorize the answers to the homework questions (Training Data) but don't understand the concepts, they will fail the final exam (Test Data) which has new questions.
-   If they understand the underlying patterns, they will succeed on both.

To ensure our models are learning patterns and not just memorizing, we use **Train-Test Splits** and **Cross-Validation**.

## Train-Test Splits (The Holdout Method)

The simplest way to evaluate a model is to split the dataset into two mutually exclusive parts:

1.  **Training Set** (typically 70-80%): Used to "teach" the model. The model sees the answers (labels) and adjusts its internal parameters.
2.  **Test Set** (typically 20-30%): Used to evaluate performance. The model never sees these answers during training. It makes predictions, and we compare them to the actual values.

**Important**: The split must be random to ensure both sets are representative of the overall population.

## Cross-Validation (CV)

A single train-test split has a flaw: it depends heavily on **which** specific data points end up in the test set. If the test set happens to be "easy," the model looks great. If it's "hard," the model looks bad.

**Cross-Validation** solves this by repeating the split process multiple times.

The most common method is **K-Fold Cross-Validation**:

1.  Split the data into $K$ equal parts (folds).
2.  Train the model $K$ times.
3.  Each time, use a different fold as the "Test Set" and the remaining $K-1$ folds as the "Training Set".
4.  Average the $K$ scores to get the final performance metric.

This gives a much more reliable estimate of how the model will perform in the real world.

## Practical Demonstration: California Housing

We will use the California Housing dataset to demonstrate how moving from a single split to Cross-Validation changes our perspective on model performance.

### Setup and Exploration

We load the data and focus on a single feature, `MedInc` (Median Income), to predict `MedHouseVal` (House Value). We do this to keep the model simple and interpretable.

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load data
data = fetch_california_housing(as_frame=True)
df = data.frame

# Check correlations to confirm MedInc is a good predictor
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr()[['MedHouseVal']].sort_values(by='MedHouseVal', ascending=False),
            annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation with House Value')
plt.show()

### The Train-Test Split

We split the data 80/20. Note the use of `random_state` for reproducibility.

In [ ]:
from sklearn.model_selection import train_test_split

# Select Feature (X) and Target (y)
X = df[['MedInc']] # Double brackets to keep X as a DataFrame (2D)
y = df['MedHouseVal']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")

Let's visualize the split to ensure the distribution of the target variable looks similar in both sets.

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(y_train, bins=30, color='blue', alpha=0.6)
plt.title(f'Train Set (Mean: {y_train.mean():.2f})')

plt.subplot(1, 2, 2)
sns.histplot(y_test, bins=30, color='orange', alpha=0.6)
plt.title(f'Test Set (Mean: {y_test.mean():.2f})')

plt.show()

### Single Split Evaluation

We train a Linear Regression model and check the $R^2$ score.

In [ ]:
from sklearn.linear_model import LinearRegression

# Train
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate
score = model.score(X_test, y_test)
print(f"Single Split Test R²: {score:.4f}")

### Cross-Validation Evaluation

Now, we use 5-Fold Cross-Validation. This simulates training and testing 5 distinct times on different subsets of the data.

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np

# Initialize a fresh model
model_cv = LinearRegression()

# Perform 5-Fold CV
# Note: We pass the FULL X and y here. The function handles the splitting.
cv_scores = cross_val_score(model_cv, X, y, cv=5, scoring="r2")

print(f"Individual CV Scores: {cv_scores}")
print(f"Average R²: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")

**Observation**: Notice that the scores vary! In one fold, the model performed much worse (or better, depending on random seed) than others. The **Average $R^2$** is a more honest reflection of the model's true capability than the single split score.

## Data Leakage

**Data Leakage** occurs when information from outside the training set improperly influences the model. This leads to overly optimistic performance during development that collapses when the model encounters real-world data.

Think of it like a student who accidentally sees the answer key before an exam—they score 100%, but learn nothing.

### Common Sources of Leakage

1.  **Preprocessing before splitting**: If you scale or impute missing values using the entire dataset, the test set statistics "leak" into training.
2.  **Temporal leakage**: Using future information to predict the past (e.g., using 2024 measurements to predict 2023 outcomes).
3.  **Target leakage**: Including features that are consequences of the target (e.g., `days_in_hospital` when predicting `will_be_hospitalized`).
4.  **Duplicate samples**: The same subject appearing in both train and test sets (common with multiple measurements per patient/sample).

### Example: The Wrong Way

Here we scale the data **before** splitting: a subtle but serious mistake.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# WRONG: Scaling before split leaks test set statistics into training
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # Learns mean/std from ALL data, including test!

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

### Example: The Right Way (Using Pipelines)

Scikit-learn's `Pipeline` ensures that preprocessing is fitted **only** on training data.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score

# CORRECT: Split first, then create a pipeline
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipeline bundles preprocessing + model
pipe = Pipeline([
    ('scaler', StandardScaler()),  # Will be fit ONLY on training data
    ('regressor', LinearRegression())
])

# Fit the pipeline (scaler learns from X_train only)
pipe.fit(X_train, y_train)

# Evaluate (scaler transforms X_test using train statistics)
print(f"Test R²: {pipe.score(X_test, y_test):.4f}")

# Cross-validation also handles this correctly
cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='r2')
print(f"CV R²: {cv_scores.mean():.4f}")

### Preventing Leakage: A Checklist

-   **Always split first**: No preprocessing touches the full dataset.
-   **Use Pipelines**: They enforce correct ordering automatically.
-   **Think causally**: "Would I have this feature available at prediction time?"
-   **Use GroupKFold**: When samples are related (e.g., multiple cells from the same patient), use `GroupKFold` to keep all samples from one subject in the same fold.

## Exercises

Now apply these concepts to the **Ames Housing dataset**.

### Load and Clean

The Ames dataset contains missing values (`NaN`). Linear Regression cannot handle these. We must drop rows with missing data in our selected columns.

### Standard Split

Perform an 80/20 split and train a Linear Regression model.

### Robustness Check (Cross-Validation)

Run 5-fold and 10-fold cross-validation to see if the performance holds up.

## Summary

We have moved from trusting a single "exam result" (Train-Test split) to looking at the "student's average performance" (Cross-Validation).

-   **Train-Test Split**: Good for quick checks and large datasets.
-   **Cross-Validation**: Essential for reliable evaluation, hyperparameter tuning, and smaller datasets.